# 11 - Enrichment v1 (distributed)

Demonstrates `AsyncFrameEnricher` and `AttachmentSpec` in a distributed Dask session managed entirely by **boti-dask**.

Key boti-dask APIs used:
- `apply_recommended_dask_config()` — sets tasks-shuffle, memory thresholds, timeouts
- `DataHelper.session(cluster_factory=LocalCluster, cluster_kwargs={...})` — owns full cluster lifecycle

In [1]:
import pandas as pd
from dask.distributed import LocalCluster

from boti_dask import apply_recommended_dask_config
from boti_data import AsyncFrameEnricher, AttachmentSpec, DataHelper

## Start the distributed session

`apply_recommended_dask_config()` sets tasks-shuffle (silencing p2p warnings), memory thresholds, and distributed timeouts before the cluster starts.  
Cluster kwargs are passed via `cluster_kwargs=` — `DaskSession` owns the full lifecycle.

In [2]:
with apply_recommended_dask_config():
    with DataHelper.session(
        cluster_factory=LocalCluster,
        cluster_kwargs={"n_workers": 2, "threads_per_worker": 1, "processes": False},
        verify_connectivity=True,
    ) as client:
        print("workers:", len(client.scheduler_info()["workers"]))

workers: 2


## Define the base frame and enrichment spec

Each `AttachmentSpec` declares which columns trigger the lookup, how to call the attachment function, and how to merge the result back.

In [3]:
base = pd.DataFrame(
    {
        "customer_id": [1, 2, 3, 4],
        "status": ["active", "inactive", "active", "active"],
    }
)


async def customer_segment_attachment(ids: list[int]):
    lookup = {1: "gold", 2: "silver", 3: "gold", 4: "bronze"}
    return pd.DataFrame(
        {
            "id": ids,
            "segment": [lookup.get(value, "unknown") for value in ids],
        }
    )


enricher = AsyncFrameEnricher(
    [
        AttachmentSpec(
            key="customer_segment",
            required_cols={"customer_id"},
            attachment_fn=customer_segment_attachment,
            col_to_kwarg={"customer_id": "ids"},
            left_on=["customer_id"],
            right_on=["id"],
            drop_cols=["id"],
            max_unique_values=1000,
        )
    ]
)
base

,customer_id,status
0,1,active
1,2,inactive
2,3,active
3,4,active


## Run enrichment inside the distributed session

In [4]:
with apply_recommended_dask_config():
    with DataHelper.session(
        cluster_factory=LocalCluster,
        cluster_kwargs={"n_workers": 2, "threads_per_worker": 1, "processes": False},
    ) as client:
        enriched = await enricher.aenrich(base, cols=["customer_segment"])
        enriched_df = enriched.compute().sort_values("customer_id").reset_index(drop=True)

enriched_df

,customer_id,status,segment
0,1,active,gold
1,2,inactive,silver
2,3,active,gold
3,4,active,bronze


In [5]:
assert "segment" in enriched_df.columns
assert enriched_df["segment"].tolist() == ["gold", "silver", "gold", "bronze"]
enriched_df

,customer_id,status,segment
0,1,active,gold
1,2,inactive,silver
2,3,active,gold
3,4,active,bronze
